In [1]:
# Load packages
import os
import pandas as pd 

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4
Found bucket: id=vwb-aou-datasets-controlled-v8r9, bucketName=vwb-aou-datasets-controlled

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20261106 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
vwb-aou-datasets-controlled-v8r9 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = gs://rw-migr

# PHENOTYPE FILE

In [3]:
# Workspace bucket 
bucket=f"{os.environ['WORKSPACE_BUCKET']}/data/saige_phenotype_files/aml_biosample_age_files/EUR_phenotype_file.txt"
df = pd.read_csv(bucket, sep='\t')
# Format person id to BGEN format 
df['person_id'] = df['person_id'].astype(str) + '_' + df['person_id'].astype(str)
# Save as phenotype file 
df.to_csv(f"EUR_phenotype_file.txt", 
          sep='\t',
          index=None)

In [4]:
# Save phenotype file to workspace bucket 
!gcloud storage cp EUR_phenotype_file.txt {os.environ['WORKSPACE_BUCKET']}/data/staar/phenotype/

Copying file://EUR_phenotype_file.txt to gs://rw-migration-aou-rw-9f879fc4/data/staar/phenotype/EUR_phenotype_file.txt
  Completed files 1/1 | 37.7MiB/37.7MiB | 5.0MiB/s                             

Average throughput: 4.2MiB/s


# COMMAND 

In [18]:
%%writefile STAARpipeline_Null_Model.R
#!/usr/bin/env Rscript

##########################################################################
# INPUT
##########################################################################

# Read command-line arguments
args <- commandArgs(trailingOnly = TRUE)

# Define input/output file paths
phenotype_path <- args[1]
grm_path <- args[2]
output_path <- args[3]

# Print input/output paths
cat("PHENOTYPE:", phenotype_path, "\n")
cat("GENETIC RELATIONSHIP MATRIX:", grm_path, "\n")
cat("OUTPUT PATH:", output_path, "\n")

###########################################################################
# MAIN FUNCTION
###########################################################################

# load packages 
library(gdsfmt)
library(SeqArray)
library(SeqVarTools)
library(STAAR)
library(STAARpipeline)

# Read phenotype/covariate data
phenotype <- read.delim(
    phenotype_path,
    header = TRUE,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# Load the sparse genetic relationship matrix
grm <- get(load(grm_path))

# Check that the input data loaded correctly
print(head(phenotype))
print(head(grm))

###########################################################
# FIT NULL MODEL
###########################################################

# Fit the Gaussian null model using the sparse GRM
obj_nullmodel <- fit_null_glmmkin_Binary_SPA(
    aml_biosample_age ~ age + sex + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10,
    data = phenotype,
    kins = grm,
    use_sparse = TRUE,
    kins_cutoff = 0.022,
    id = "person_id",
    family = binomial(link = "logit"),
    verbose = TRUE
)

# Save the fitted null model to the specified output path
save(obj_nullmodel,file=paste0(output_path))

###########################################################
#   
###########################################################

Overwriting STAARpipeline_Null_Model.R


In [19]:
# Copy script in workspace bucket 
!gcloud storage cp STAARpipeline_Null_Model.R ${WORKSPACE_BUCKET}/data/scripts/

Copying file://STAARpipeline_Null_Model.R to gs://rw-migration-aou-rw-9f879fc4/data/scripts/STAARpipeline_Null_Model.R
  Completed files 1/1 | 1.8kiB/1.8kiB                                          


In [20]:
# Check script in workspace bucket 
!gcloud storage cat ${WORKSPACE_BUCKET}/data/scripts/STAARpipeline_Null_Model.R 

#!/usr/bin/env Rscript

##########################################################################
# INPUT
##########################################################################

# Read command-line arguments
args <- commandArgs(trailingOnly = TRUE)

# Define input/output file paths
phenotype_path <- args[1]
grm_path <- args[2]
output_path <- args[3]

# Print input/output paths
cat("PHENOTYPE:", phenotype_path, "\n")
cat("GENETIC RELATIONSHIP MATRIX:", grm_path, "\n")
cat("OUTPUT PATH:", output_path, "\n")

###########################################################################
# MAIN FUNCTION
###########################################################################

# load packages 
library(gdsfmt)
library(SeqArray)
library(SeqVarTools)
library(STAAR)
library(STAARpipeline)

# Read phenotype/covariate data
phenotype <- read.delim(
    phenotype_path,
    header = TRUE,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

# Load the sparse genetic relationship matrix
grm 

# SUBMIT COMMAND 

In [21]:
import shlex

# Logging directory
LOGGING = f"{os.environ['WORKSPACE_BUCKET']}/data/logging"

# Dsub script
SCRIPT = "STAARpipeline_Null_Model.R"
PATH = f"{os.environ['WORKSPACE_BUCKET']}/data/scripts/{SCRIPT}"

# Disk size
DISK_SIZE = 10

# File paths
PHENOTYPE = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/phenotype/EUR_phenotype_file.txt"
GRM = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/grm/sparseGRM.RData"
OUTPUT = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/null_modle/obj_spa_nullmodel.Rdata"

# R script command
R_COMMAND = 'Rscript "$SCRIPT" "$PHENOTYPE" "$GRM" "$OUTPUT"'

command = [
    "aou_dsub",
    "--image", f"{os.environ['ARTIFACT_REGISTRY_DOCKER_REPO']}/zilinli/staarpipeline:0.9.7",
    "--boot-disk-size", "5",
    "--disk-size", str(DISK_SIZE),
    "--machine-type", "n1-standard-2",
    "--logging", LOGGING,
    "--input", f"SCRIPT={PATH}",
    "--input", f"PHENOTYPE={PHENOTYPE}",
    "--input", f"GRM={GRM}",
    "--output", f"OUTPUT={OUTPUT}",
    "--command", R_COMMAND,
]

# Run in same shell as source command
dsub_command = subprocess.run(
    f"source ~/aou_dsub.bash && {shlex.join(command)}",
    shell=True,
    executable="/bin/bash",
    capture_output=True,
    text=True
)

print(dsub_command.stdout)
print(dsub_command.stderr)

rscript--vienose--261107-013812-65

Job properties:
  job-id: rscript--vienose--261107-013812-65
  job-name: rscript
  user-id: vienose
Provider internal-id (operation): projects/wb-fast-radish-6415/locations/us-central1/jobs/rscript--vienose--261107-013812-65-0-0
Launched job-id: rscript--vienose--261107-013812-65
To check the status, run:
  dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'rscript--vienose--261107-013812-65' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'rscript--vienose--261107-013812-65' --users 'vienose'

